In [1]:
import sympy as smp
import numpy as np

In [2]:
t, m, R, g, l, lm1, lm2 = smp.symbols(r"t m R g l \lambda_1 \lambda_2", real=True)
r, the, phi = smp.symbols(r"r \theta \phi", cls=smp.Function)
r = r(t)
the = the(t)
phi = phi(t)

def dt(x):
    return smp.diff(x), smp.diff(smp.diff(x))

r_d, r_dd = dt(r)
the_d, the_dd = dt(the)
phi_d, phi_dd = dt(phi)

In [3]:
v = smp.Matrix([r_d, r*the_d])

In [4]:
I = 1/2 * m * l**2
T = 1/2 * m * v.dot(v) + 1/2 * I * phi_d**2
U = m * g * r * smp.cos(the)
L = T - U
L

-g*m*r(t)*cos(\theta(t)) + 0.25*l**2*m*Derivative(\phi(t), t)**2 + 0.5*m*(r(t)**2*Derivative(\theta(t), t)**2 + Derivative(r(t), t)**2)

In [5]:
def lagrange_eq(L, variables, R=0):
    lagrange_equations = []

    for q in variables:
        q_dot = smp.diff(q, t)
        LE = smp.diff(smp.diff(L, q_dot), t) - smp.diff(L, q) + smp.diff(R, q_dot)
        LE = LE.simplify()
        lagrange_equations.append(LE)
        
    return tuple(lagrange_equations)

In [6]:
LEr, LEthe, LEphi = lagrange_eq(L, (r, the, phi))

In [7]:
LEr

1.0*m*(g*cos(\theta(t)) - r(t)*Derivative(\theta(t), t)**2 + Derivative(r(t), (t, 2)))

In [8]:
LEthe

m*(-g*sin(\theta(t)) + 1.0*r(t)*Derivative(\theta(t), (t, 2)) + 2.0*Derivative(\theta(t), t)*Derivative(r(t), t))*r(t)

In [9]:
LEphi

0.5*l**2*m*Derivative(\phi(t), (t, 2))

# Constraints

The constraint the disc stays anchored to the hoop. So the position $r$ is always equal to the sum of the two radiuses $R + l$.

$$ r = R + l $$
$$ r - R - l = 0 $$

And the disc has no slipping on the surface.

$$ l d\phi - R d\theta = 0 $$

$$ l \dot{\phi} - R \dot{\theta} = 0 $$

In [10]:
system = smp.Matrix(
    [
        LEr + lm1,
        LEthe + lm2 * (-R),
        LEphi + lm2 * (l),
    ]
)
system

Matrix([
[                                   \lambda_1 + 1.0*m*(g*cos(\theta(t)) - r(t)*Derivative(\theta(t), t)**2 + Derivative(r(t), (t, 2)))],
[-R*\lambda_2 + m*(-g*sin(\theta(t)) + 1.0*r(t)*Derivative(\theta(t), (t, 2)) + 2.0*Derivative(\theta(t), t)*Derivative(r(t), t))*r(t)],
[                                                                                 \lambda_2*l + 0.5*l**2*m*Derivative(\phi(t), (t, 2))]])

Substitue the constraints

In [11]:
system = system.subs({r: R + l, r_d: 0, r_dd: 0, phi_dd: the_dd})
system

Matrix([
[              \lambda_1 + 1.0*m*(g*cos(\theta(t)) - (R + l)*Derivative(\theta(t), t)**2)],
[-R*\lambda_2 + m*(R + l)*(-g*sin(\theta(t)) + 1.0*(R + l)*Derivative(\theta(t), (t, 2)))],
[                                  \lambda_2*l + 0.5*l**2*m*Derivative(\theta(t), (t, 2))]])

In [12]:
sols = smp.solve(system, [lm1], simplify=False)
sols

{\lambda_1: R*m*Derivative(\theta(t), t)**2 - g*m*cos(\theta(t)) + l*m*Derivative(\theta(t), t)**2}

In [13]:
sols[lm1]

R*m*Derivative(\theta(t), t)**2 - g*m*cos(\theta(t)) + l*m*Derivative(\theta(t), t)**2

In [14]:
smp.solve(system[1], lm2)[0]

m*(R*(R*Derivative(\theta(t), (t, 2)) - g*sin(\theta(t)) + 2.0*l*Derivative(\theta(t), (t, 2))) - g*l*sin(\theta(t)) + l**2*Derivative(\theta(t), (t, 2)))/R

In [15]:
smp.solve(system[2].subs({lm2: smp.solve(system[1], lm2)[0]}), the_dd)[0]

2.0*g*(R + l)*sin(\theta(t))/(2.0*R**2 + 5.0*R*l + 2.0*l**2)

We obtain that

$$ \ddot{\theta} = a \sin(\theta)$$

We can use the chain rule $\ddot{\theta} = \frac{d\dot{\theta}}{d\theta} \frac{d\theta}{dt} = \dot{\theta} \frac{d\dot{\theta}}{d\theta}$

$$ \int \dot{\theta} d\dot{\theta} = \int a \sin(\theta) d\theta $$

$$ \frac{1}{2}\dot{\theta}^2 = - a \cos(\theta) + c $$

$$ \dot{\theta}^2 = -2 a \cos(\theta) + c $$

Solve for $c$, such that the intial conditions are $\theta = \dot{\theta} = 0$

$$ c = 2a $$

$$ \dot{\theta}^2 = 2a(1 - \cos(\theta)) $$

In [16]:
a = smp.solve(system[2].subs({lm2: smp.solve(system[1], lm2)[0]}), the_dd)[0].subs({smp.sin(the): 1})
a

2.0*g*(R + l)/(2.0*R**2 + 5.0*R*l + 2.0*l**2)

We can substitute $\dot{\theta}^2$ inside the $r$ equation.

In [17]:
lm1_eq = system[0].subs({the_d**2: 2 * a * (1 - smp.cos(the))}).simplify()
lm1_eq

\lambda_1 + 1.0*m*(4.0*g*(R + l)**2*(cos(\theta(t)) - 1)/(2.0*R**2 + 5.0*R*l + 2.0*l**2) + g*cos(\theta(t)))

In [18]:
lm1_eq = smp.solve(lm1_eq, lm1)[0]
smp.Eq(lm1, lm1_eq)

Eq(\lambda_1, g*m*(-6.0*R**2*cos(\theta(t)) + 4.0*R**2 - 13.0*R*l*cos(\theta(t)) + 8.0*R*l - 6.0*l**2*cos(\theta(t)) + 4.0*l**2)/(2.0*R**2 + 5.0*R*l + 2.0*l**2))

This is force of constraint along the $r$ direction. We need to find when this equation equals zero

In [19]:
smp.Eq(0, lm1_eq).simplify()

Eq(g*m*(-6.0*R**2*cos(\theta(t)) + 4.0*R**2 - 13.0*R*l*cos(\theta(t)) + 8.0*R*l - 6.0*l**2*cos(\theta(t)) + 4.0*l**2)/(2.0*R**2 + 5.0*R*l + 2.0*l**2), 0)

In [20]:
smp.solve(smp.Eq(0, lm1_eq), the)[1]

acos(4.0*(R + l)**2/((2.0*R + 3.0*l)*(3.0*R + 2.0*l)))